In [68]:
%reload_ext autoreload
%autoreload 2

In [69]:
from waymo_agent import *

In [70]:
# from kret_studies import *
# from kret_studies.notebook import *
# from kret_studies.complex import *

# logger = get_notebook_logger()

In [71]:
from waymo_agent import *
from waymo_agent.osmnx import *
from waymo_agent.data_classes import *
from waymo_agent.action_heuristic import *
from waymo_agent.graph_env import *
from waymo_agent.simulation import *

#### Load ENV

In [72]:
envconfig = EnvConfig()
cfg = envconfig

In [73]:
env = RideShareEnv(cfg)
G = env.graph

/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/Users/Akseldkw/coding/Columbia/RL-Project/waymo_agent/graph_env/mixin/obs_space_mixin.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  requests = RequestDF(pd.concat([real_requests, filler_requests], ignore_index=True).reset_index(drop=True))
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gy

In [74]:
s, info = env.reset()

/Users/Akseldkw/coding/Columbia/RL-Project/waymo_agent/graph_env/mixin/obs_space_mixin.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  requests = RequestDF(pd.concat([real_requests, filler_requests], ignore_index=True).reset_index(drop=True))
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/gymnasium/spaces/box.py:424: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


In [75]:
veh = env.observation_curr["vehicles"]
req = env.observation_curr["pending_requests"]
rides = env.observation_curr["active_rides"]
type(veh), type(req), type(rides)

(waymo_agent.data_classes.vehicles.VehicleDF,
 waymo_agent.data_classes.requests.RequestDF,
 waymo_agent.data_classes.active_rides.ActiveRideDF)

In [76]:
inspect_graph(G)

Graph has 782 nodes and 2746 edges.
Sample node -sample_node_id=42432703- attributes: {'y': 40.7603553, 'x': -73.9912267, 'highway': 'traffic_signals', 'street_count': 4, 'lambda': 0.0, 'x_centered': -0.011592, 'y_centered': 0.005372, 'x_norm': -0.181407, 'y_norm': 0.08407}

Sample edge attributes: {'osmid': 1023818802, 'highway': 'primary', 'lanes': "('4', '3')", 'maxspeed': 25.0, 'name': 'Amsterdam Avenue', 'oneway': False, 'reversed': False, 'length': 80.8967000992589, 'geometry': <LINESTRING (-73.985 40.774, -73.985 40.774, -73.985 40.774, -73.984 40.775,...>, 'travel_time_minutes': 0.1942}

cls.edge_length_unit='meters', cls.speed_unit='km/h', cls.time_unit='minutes'


In [77]:
node_df = env.node_df

In [78]:
node_df

,node_id,y,x,street_count,lambda,x_centered,y_centered,x_norm,y_norm
0,42421728,40.798048,-73.960044,3,0.0002,0.019591,0.043065,0.306596,0.673944
1,42421737,40.799244,-73.962873,4,0.0001,0.016762,0.044260,0.262312,0.692660
2,42421741,40.800429,-73.965691,4,0.0002,0.013944,0.045446,0.218219,0.711217
3,42421745,40.801398,-73.967996,4,0.0004,0.011639,0.046415,0.182153,0.726375
4,42421951,40.703326,-74.007775,4,0.0000,-0.028140,-0.051657,-0.440376,-0.808412
...,...,...,...,...,...,...,...,...,...
777,12554607194,40.712465,-74.004865,3,0.0001,-0.025230,-0.042519,-0.394840,-0.665404
778,13099951760,40.769688,-73.994623,3,0.0004,-0.014988,0.014705,-0.234550,0.230125
779,13205862060,40.732353,-73.984937,4,0.0004,-0.005302,-0.022630,-0.082982,-0.354156
780,13208007103,40.772890,-73.982107,4,0.0002,-0.002472,0.017907,-0.038693,0.280237


In [79]:
xs = node_df["x"].to_numpy()
ys = node_df["y"].to_numpy()

In [80]:
x0 = xs.mean()
y0 = ys.mean()
xs_c = xs - x0
ys_c = ys - y0
max_abs = max(np.abs(xs_c).max(), np.abs(ys_c).max())  # largest radius

In [81]:
y_norm = ys_c / max_abs

In [82]:
from kret_sandbox.VIS import dtt

In [83]:
y_orig = node_df["y_norm"] * max_abs + y0

In [84]:
dtt([node_df, y_norm, y_orig])

node_id 
 y 
 x 
 street_count 
 lambda 
 x_centered 
 y_centered 
 x_norm 
 y_norm 
 
 
 
 
 3 
 42421745 
 40.801 
 -73.968 
 4 
 0.0 
 0.012 
 0.046 
 0.182 
 0.726 
 
 
 108 
 42430271 
 40.743 
 -73.993 
 4 
 0.0 
 -0.013 
 -0.012 
 -0.206 
 -0.189 
 
 
 592 
 1918039864 
 40.724 
 -73.993 
 4 
 0.0 
 -0.013 
 -0.031 
 -0.204 
 -0.483 
 
 
 617 
 3785532382 
 40.780 
 -73.944 
 4 
 0.0 
 0.036 
 0.025 
 0.556 
 0.391 
 
 
 741 
 9168918373 
 40.797 
 -73.976 
 3 
 0.0 
 0.004 
 0.042 
 0.063 
 0.661 
 
 
 
 
 
 
 0 
 
 
 
 
 3 
 0.726 
 
 
 108 
 -0.189 
 
 
 592 
 -0.483 
 
 
 617 
 0.391 
 
 
 741 
 0.661 
 
 
 
 
 
 
 y_norm 
 
 
 
 
 3 
 40.801 
 
 
 108 
 40.743 
 
 
 592 
 40.724 
 
 
 617 
 40.780 
 
 
 741 
 40.797

In [85]:
from waymo_agent.osmnx.euclidean_L2_embed import x_y_normed_to_orig

In [89]:
lon, lat = x_y_normed_to_orig(env.l2_recovery, veh.loc_x_norm, veh.loc_y_norm)

In [90]:
lon

array([-73.97719602, -73.96940657])

In [91]:
lat

array([40.76432402, 40.7577077 ])